In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
restaurants = {
    "Bella Italia Pune": {
        "cuisine": "Italian",
        "city": "Pune",
        "rating": 4.4,
        "open_days": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"],
        "cost_for_two": 1200
    },
    "Olive Bistro Pune": {
        "cuisine": "Italian",
        "city": "Pune",
        "rating": 4.2,
        "open_days": ["Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
        "cost_for_two": 1600
    },
    "Spice Garden Pune": {
        "cuisine": "Indian",
        "city": "Pune",
        "rating": 4.1,
        "open_days": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
        "cost_for_two": 800
    },
    "Pasta Corner Mumbai": {
        "cuisine": "Italian",
        "city": "Mumbai",
        "rating": 4.3,
        "open_days": ["Monday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
        "cost_for_two": 1400
    }
}

print("Sample restaurant database created.")
print("Number of sample restaurants:", len(restaurants))

Sample restaurant database created.
Number of sample restaurants: 4


In [3]:
@tool
def find_restaurants(cuisine: str, city: str) -> str:
    """Find sample restaurants matching a cuisine and city."""

    matches = [
        name
        for name, details in restaurants.items()
        if details["cuisine"].casefold() == cuisine.strip().casefold()
        and details["city"].casefold() == city.strip().casefold()
    ]

    if not matches:
        return f"No sample {cuisine} restaurants found in {city}."

    return (
        f"Sample restaurants matching {cuisine} in {city}: "
        + ", ".join(matches)
    )


print(find_restaurants.invoke({
    "cuisine": "Italian",
    "city": "Pune"
}))

Sample restaurants matching Italian in Pune: Bella Italia Pune, Olive Bistro Pune


In [4]:
@tool
def get_restaurant_rating(restaurant_name: str) -> str:
    """Get the sample rating out of 5 for a restaurant."""

    details = restaurants.get(restaurant_name.strip())

    if details is None:
        return f"No sample rating found for {restaurant_name}."

    return (
        f"Restaurant: {restaurant_name}\n"
        f"Sample rating: {details['rating']}/5"
    )


print(get_restaurant_rating.invoke({
    "restaurant_name": "Bella Italia Pune"
}))


Restaurant: Bella Italia Pune
Sample rating: 4.4/5


In [5]:
@tool
def check_restaurant_open(restaurant_name: str, day: str) -> str:
    """Check whether a sample restaurant is listed as open on a given weekday."""

    details = restaurants.get(restaurant_name.strip())

    if details is None:
        return f"No sample opening information found for {restaurant_name}."

    day = day.strip().capitalize()

    valid_days = {
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    }

    if day not in valid_days:
        return "Invalid day. Use a weekday name such as Thursday."

    status = (
        "listed as open"
        if day in details["open_days"]
        else "listed as closed"
    )

    return f"{restaurant_name} is {status} on {day} in the sample data."


print(check_restaurant_open.invoke({
    "restaurant_name": "Bella Italia Pune",
    "day": "Thursday"
}))

Bella Italia Pune is listed as open on Thursday in the sample data.


In [6]:
@tool
def estimate_cost_for_two(restaurant_name: str) -> str:
    """Get the sample estimated cost for two people at a restaurant."""

    details = restaurants.get(restaurant_name.strip())

    if details is None:
        return f"No sample cost information found for {restaurant_name}."

    return (
        f"Restaurant: {restaurant_name}\n"
        f"Sample estimated cost for two: "
        f"₹{details['cost_for_two']:,.2f}"
    )


print(estimate_cost_for_two.invoke({
    "restaurant_name": "Bella Italia Pune"
}))

Restaurant: Bella Italia Pune
Sample estimated cost for two: ₹1,200.00


In [7]:
tools = [
    find_restaurants,
    get_restaurant_rating,
    check_restaurant_open,
    estimate_cost_for_two
]

tool_map = {
    item.name: item for item in tools
}

llm_with_tools = llm.bind_tools(tools)

print("Four tools bound in a single bind_tools() call:")

for item in tools:
    print("-", item.name)

Four tools bound in a single bind_tools() call:
- find_restaurants
- get_restaurant_rating
- check_restaurant_open
- estimate_cost_for_two


In [8]:
user_query = """
Recommend a good Italian restaurant in Pune for Thursday.

Use the available tools to:
1. Find Italian restaurants in Pune.
2. Check the rating of a matching restaurant.
3. Check whether it is open on Thursday.
4. Estimate the cost for two.

Try to use all four tools before giving your recommendation.
Use only the returned data and clearly identify it as sample data.
"""

messages = [HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")

for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest any tool calls. Run this cell again."
    )

User query:

Recommend a good Italian restaurant in Pune for Thursday.

Use the available tools to:
1. Find Italian restaurants in Pune.
2. Check the rating of a matching restaurant.
3. Check whether it is open on Thursday.
4. Estimate the cost for two.

Try to use all four tools before giving your recommendation.
Use only the returned data and clearly identify it as sample data.


Model-suggested tool calls:
Tool: find_restaurants
Arguments: {'city': 'Pune', 'cuisine': 'Italian'}


In [9]:
messages.append(response)

tool_results = []
executed_tools = set()

# Execute the model-suggested calls first
for call in response.tool_calls:
    tool_name = call["name"]
    tool_args = call["args"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    result = tool_map[tool_name].invoke(tool_args)

    tool_results.append((tool_name, result))
    executed_tools.add(tool_name)

    print(f"\nModel-suggested tool executed: {tool_name}")
    print("Arguments:", tool_args)
    print("Result:")
    print(result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

# Run any missing lookups manually using the sample restaurant
# selected for this exercise.
manual_calls = [
    ("find_restaurants", {
        "cuisine": "Italian",
        "city": "Pune"
    }),
    ("get_restaurant_rating", {
        "restaurant_name": "Bella Italia Pune"
    }),
    ("check_restaurant_open", {
        "restaurant_name": "Bella Italia Pune",
        "day": "Thursday"
    }),
    ("estimate_cost_for_two", {
        "restaurant_name": "Bella Italia Pune"
    })
]

for tool_name, tool_args in manual_calls:
    if tool_name not in executed_tools:
        result = tool_map[tool_name].invoke(tool_args)

        tool_results.append((tool_name, result))
        executed_tools.add(tool_name)

        print(f"\nMissing lookup executed manually: {tool_name}")
        print("Arguments:", tool_args)
        print("Result:")
        print(result)

print("\nAll four restaurant lookups have been executed.")


Model-suggested tool executed: find_restaurants
Arguments: {'city': 'Pune', 'cuisine': 'Italian'}
Result:
Sample restaurants matching Italian in Pune: Bella Italia Pune, Olive Bistro Pune

Missing lookup executed manually: get_restaurant_rating
Arguments: {'restaurant_name': 'Bella Italia Pune'}
Result:
Restaurant: Bella Italia Pune
Sample rating: 4.4/5

Missing lookup executed manually: check_restaurant_open
Arguments: {'restaurant_name': 'Bella Italia Pune', 'day': 'Thursday'}
Result:
Bella Italia Pune is listed as open on Thursday in the sample data.

Missing lookup executed manually: estimate_cost_for_two
Arguments: {'restaurant_name': 'Bella Italia Pune'}
Result:
Restaurant: Bella Italia Pune
Sample estimated cost for two: ₹1,200.00

All four restaurant lookups have been executed.


In [10]:
tool_results_text = "\n\n".join(
    f"{name}:\n{result}"
    for name, result in tool_results
)

final_prompt = HumanMessage(
    content=(
        "Act as a restaurant recommendation assistant. Use the tool "
        "results below to recommend a suitable Italian restaurant in Pune "
        "for Thursday. Include the rating, opening status, and estimated "
        "cost for two if available. Do not invent details or alter values. "
        "Clearly state that the information is sample data, not verified "
        "live restaurant information.\n\n"
        f"Tool results:\n{tool_results_text}"
    )
)

final_response = llm.invoke([final_prompt])

print("Final restaurant recommendation:")
print(final_response.content)

Final restaurant recommendation:
**Restaurant Recommendation (Sample Data Only – Not Verified Live Information)**  

**Bella Italia Pune**  
- **Cuisine:** Italian  
- **Rating:** 4.4 / 5 (sample rating)  
- **Opening status for Thursday:** Open (as listed in the sample data)  
- **Estimated cost for two:** ₹1,200.00 (sample estimate)  

Based on the sample data provided, Bella Italia Pune appears to be a solid choice for an Italian meal on Thursday, with a strong rating, confirmed opening status, and a reasonable cost for two diners.  

*Please note that all details above are drawn from sample data and have not been verified against live restaurant information.*


In [11]:
required_tools = {
    "find_restaurants",
    "get_restaurant_rating",
    "check_restaurant_open",
    "estimate_cost_for_two"
}

model_suggested_tools = {
    call["name"] for call in response.tool_calls
}

executed_tools = {
    name for name, result in tool_results
}

print("Model-suggested tools:", model_suggested_tools)
print("Executed tools:", executed_tools)

assert required_tools.issubset(executed_tools), (
    "One or more restaurant tools were not executed."
)

assert final_response.content.strip(), (
    "The final recommendation is empty."
)

print("Validation passed.")
print("All four tools were executed.")
print("The final restaurant recommendation was generated.")

if required_tools.issubset(model_suggested_tools):
    print("The model suggested all four tools.")
else:
    print(
        "Some tools were executed manually because the model "
        "did not suggest every tool."
    )

Model-suggested tools: {'find_restaurants'}
Executed tools: {'get_restaurant_rating', 'find_restaurants', 'estimate_cost_for_two', 'check_restaurant_open'}
Validation passed.
All four tools were executed.
The final restaurant recommendation was generated.
Some tools were executed manually because the model did not suggest every tool.


In [12]:
print("Lab 6 - Exercise 10 completed.")
print("Four @tool functions were created.")
print("All four tools were bound in one bind_tools() call.")
print("Model-suggested and manually executed calls were recorded.")
print("A final restaurant recommendation was generated.")
print("All restaurant information is sample data.")

Lab 6 - Exercise 10 completed.
Four @tool functions were created.
All four tools were bound in one bind_tools() call.
Model-suggested and manually executed calls were recorded.
A final restaurant recommendation was generated.
All restaurant information is sample data.
